### Import statements

In [ ]:
import os
import csv
import re
import git
import bisect
from collections import Counter
from copy import deepcopy
import requests
from lxml import etree
import logging
import pandas as pd
import spacy
from MyCapytain.resolvers.cts.local import CtsCapitainsLocalResolver
from MyCapytain.resources.prototypes.metadata import UnknownCollection
from dicesapi import DicesAPI

### Global values

In [ ]:
data_dir = "data"
git_base_url = "http://github.com/PerseusDL"
repo_names = ["canonical-greekLit"]
nsmap = {
    "cts": "http://chs.harvard.edu/xmlns/cts",
    "tei": "http://www.tei-c.org/ns/1.0",
    "py": "http://codespeak.net/lxml/objectify/pytype",
}
spacy_model = "grc_odycy_joint_trf"

### Create local clone of Perseus repository

In [ ]:
print('Checking for local text repositories...')
for repo in repo_names:
    dest_path = os.path.join(data_dir, repo)
    remote_url = f"{git_base_url}/{repo}.git"

    if os.path.exists(dest_path):
        print(f' - {dest_path} exists!')
    else:
        print(f' - retrieving {dest_path}')
        git.Repo.clone_from(remote_url, dest_path)

### Read text metadata

In [ ]:
corpus_file = os.path.join(data_dir, "corpus.tsv")
print(f"Reading text metadata from {corpus_file}")

corpus = []

with open(corpus_file) as f:
    reader = csv.DictReader(f, delimiter="\t")
    for rec in reader:
        corpus.append(dict(
            author = rec["author"],
            title = rec["title"],
            urn = rec["urn"],
            prefix = rec["prefix"],
        ))
print(f" - got data on {len(corpus)} texts")

### Load XML using CTS API

#### initialize local resolver

In [ ]:
repo_paths = [os.path.join(data_dir, repo) for repo in repo_names]
resolver = CtsCapitainsLocalResolver(repo_paths)

#### retrieve the texts

In [ ]:
print("Retrieving texts...")
for text in corpus:
    try:
        text["xml"] = resolver.getTextualNode(text["urn"], text["prefix"]).xml
    except UnknownCollection:
        print(f" - {text['author']} {text['title']} {text['prefix']} failed")

print(len([text for text in corpus if text.get("xml") is not None]), "texts retrieved")

### Extract verse lines

#### build an array of lines

In [ ]:
print("Building line arrays...")
for i, text in enumerate(corpus):
    print(f"[{i+1}/{len(corpus)}]", text["author"], text["title"], text["prefix"], end="\t")

    # bail if no XML
    if text.get("xml") is None:
        print("failed: no XML")
        continue

    # initialize line array
    text['line_array'] = []

    # work from copy of xml
    xml = deepcopy(text["xml"])
        
    # remove notes
    for note in xml.findall(".//tei:note", namespaces=nsmap):
        note.clear(keep_tail=True)
        
    # remove deleted lines
    for del_ in xml.findall(".//tei:del", namespaces=nsmap):
        del_.clear(keep_tail=True)
        
    # iterate over verse lines
    for l in xml.findall(".//tei:l", namespaces=nsmap):
        line_num = l.get("n")
        if line_num is None:
            continue
        
        line_text = "".join(s for s in l.itertext())
        line_text = re.sub(r"\s+", " ", line_text).strip()
        text['line_array'].append(dict(
            n = line_num,
            text = line_text,
        ))

    print(len(text['line_array']), "lines")

#### index lines by starting character position

In [ ]:
print("Building line indices...")
success = 0

for text in corpus:
    # start index at zero
    text["line_index"] = []
    cumsum = 0

    # iterate over line array, add length (plus 1 for spaces between lines)
    for line in text["line_array"]:
        text["line_index"].append(cumsum)
        cumsum += len(line["text"]) + 1

    # make sure the count works out
    if cumsum != len(" ".join(line["text"] for line in text["line_array"])) + 1:
        print(" - character count doesn't match:", text["author"], text["title"], text["prefix"])
    else:
        success += 1

print(success, " indices built successfully")

### Parse text with spaCy
#### load spacy model

In [ ]:
nlp = spacy.load(spacy_model)

#### parse each text as one long string

In [ ]:
print(f"Parsing with {spacy_model}...")

for i, text in enumerate(corpus):
    print(f"[{i+1}/{len(corpus)}]", text["author"], text["title"], text["prefix"], end="\t")

    one_long_string = " ".join(line["text"] for line in text["line_array"])
    text["spacy_doc"] = nlp(one_long_string)

    print(len(text["spacy_doc"]), "tokens")
    

### Create token table

In [ ]:
tokens = []

for text in corpus:
    for token in text["spacy_doc"]:
        i = bisect.bisect_right(text["line_index"], token.idx) - 1
        locus = text["line_array"][i]["n"]
        if locus is None:
            continue
            
        if text["prefix"] is not None:
            locus = text["prefix"] + "." + locus

        tokens.append(dict(
            urn = text["urn"] + ":" + locus,
            author = text["author"],
            work = text["title"],
            pref = text["prefix"],
            line = text["line_array"][i]["n"],
            token = token.text,
            lemma = token.lemma_,
            pos = token.pos_,
            verbform = ";".join(token.morph.get("VerbForm")),                
            mood = ";".join(token.morph.get("Mood")),
            tense = ";".join(token.morph.get("Tense")),                
            voice = ";".join(token.morph.get("Voice")),
            person = ";".join(token.morph.get("Person")),
            number = ";".join(token.morph.get("Number")),
            case = ";".join(token.morph.get("Case")),
            gender = ";".join(token.morph.get("Gender")),
        ))

tokens = pd.DataFrame(tokens)

### Identify speech/narration

#### instantiate API

In [ ]:
api = DicesAPI()

#### download full speech list

In [ ]:
all_speeches = api.getSpeeches()
print(len(all_speeches), "speeches")

#### edit two speeches

This is necessary to fix line differences between editions used by DICES and Perseus

In [ ]:
for s in all_speeches:
    # missing line number in Apollonius
    if s.work.urn=="urn:cts:greekLit:tlg0001.tlg001.perseus-grc2" and s.l_la=="3.739":
        s.l_la = "3.738"

    # missing line number in Odyssey
    if s.work.urn=="urn:cts:greekLit:tlg0012.tlg002.perseus-grc2" and s.l_fi=="10.456":
        s.l_fi = "10.455"

#### create a table of all line numbers in order

In [ ]:
all_lines = []
for text in corpus:
    for line in text["line_array"]:
        locus = line["n"]
        if locus is None:
            continue
        if text["prefix"] is not None:
            locus = text["prefix"] + "." + locus
        all_lines.append(dict(
            urn = text["urn"] + ":" + locus,
            speech_id = None,
            speaker = None,
            addressee = None,
            level = 0,
        ))
all_lines = pd.DataFrame(all_lines)

#### mark all lines that belong to speeches

In [ ]:
urns = [text["urn"] for text in corpus]

for speech in all_speeches:
    
    # skip speeches that don't appear in our corpus
    if speech.work.urn not in urns:
        continue

    # skip speeches that span more than one book
    if "." in speech.l_fi:
        first_prefix, first_line = speech.l_fi.rsplit(".", 1)
    else:
        first_prefix, last_line = None, speech.l_fi

    if "." in speech.l_la:
        last_prefix, last_line = speech.l_la.rsplit(".", 1)
    else:
        last_prefix, last_line = None, speech.l_la

    if first_prefix != last_prefix:
        continue

    # determine first and last rows in table
    i_first = all_lines.index[
        all_lines["urn"].eq(speech.work.urn + ":" + speech.l_fi)
        ][0]
    i_last = all_lines.index[
        all_lines["urn"].eq(speech.work.urn + ":" + speech.l_la)
        ][0]
    
    # set speech attributes for all rows in between
    all_lines.loc[i_first:i_last, "speech_id"] = speech._attributes["public_id"]
    all_lines.loc[i_first:i_last, "speaker"] = speech.getSpkrString()
    all_lines.loc[i_first:i_last, "addressee"] = speech.getAddrString()
    all_lines.loc[i_first:i_last, "level"] = speech.level + 1
    

### Merge line table with token table to link tokens with speeches

In [ ]:
df = pd.merge(all_lines, tokens, on="urn")

### Export token table

In [ ]:
output_file = os.path.join(data_dir, "tokens.tsv")
df.to_csv(output_file, index=False, sep="\t")
print(f"{len(df)} tokens exported to {output_file}")

In [ ]:
nar = Counter(df.loc[df["speaker"].isna()]["lemma"])
nar.most_common(25)

In [ ]:
sp = Counter(df.loc[~df["speaker"].isna()]["lemma"])
sp.most_common(25)